In [7]:
import json
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from helpers.triplets import TripletGenerator
from llama_index.llms.openai import OpenAI
from llama_index.core import PropertyGraphIndex
from llama_index.core.graph_stores import SimplePropertyGraphStore
from llama_index.core import Settings
from IPython.display import Markdown, display
from dotenv import load_dotenv
from llama_index.core import StorageContext
from typing import List, Tuple, Dict
import os
from tqdm import tqdm
from datetime import datetime
import re
from llama_index.core.schema import BaseNode, MetadataMode, TextNode, TransformComponent


load_dotenv("devops/env/default.env")
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

with open('btc_blocks.json', 'r') as f:
    blocks_data = json.load(f)

with open('economic_indicators.json', 'r') as f:
    economic_data = json.load(f)

with open('on_chain_metrics.json', 'r') as f:
    onchain_data = json.load(f)


tg = TripletGenerator()
nodes, relations, text_nodes = tg.load_and_process_data(blocks_data, economic_data, onchain_data)

embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
llm = OpenAI(model="gpt-4.1-mini", 
             temperature=0,
             api_key=OPENAI_API_KEY)
Settings.llm = llm
Settings.embed_model = embed_model

# based on BaseNode embedding texts
node_texts = []
for node in nodes:
    node_texts.append("\n".join([f"{key}: {node.properties[key]}" for key in node.properties.keys()]))
    

node_embeddings = embed_model.get_text_embedding_batch(node_texts)
#text_embeddings = embed_model.get_text_embedding_batch([text_node.text for text_node in text_nodes])
for node, embedding in zip(nodes, node_embeddings):
    node.embedding = embedding
# # for text_node, embedding in zip(text_nodes, text_embeddings):
# #     text_node.embedding = embedding


2025-04-29 17:17:15,256 - helpers.triplets - INFO - Starting data processing for property graph generation...
2025-04-29 17:17:15,256 - helpers.triplets - INFO - Processing 1 blocks...
2025-04-29 17:17:15,267 - helpers.triplets - INFO - Processing economic indicators (7 indicators)...


2025-04-29 17:17:15,272 - helpers.triplets - INFO - Processing on-chain metrics (8 metrics)...
2025-04-29 17:17:18,254 - helpers.triplets - INFO - Creating cross-domain relationships...
2025-04-29 17:17:24,279 - helpers.triplets - INFO - Creating domain-specific relationships...
2025-04-29 17:17:24,280 - helpers.triplets - INFO - Generating text nodes for embedding...
2025-04-29 17:17:24,356 - helpers.triplets - INFO - Generated 8658 text nodes for embedding
2025-04-29 17:17:24,357 - helpers.triplets - INFO - Property graph generation complete:
2025-04-29 17:17:24,357 - helpers.triplets - INFO -   - 6810 nodes created
2025-04-29 17:17:24,358 - helpers.triplets - INFO -   - 1848 relations created
2025-04-29 17:17:24,358 - helpers.triplets - INFO -   - 8658 text nodes created for embedding
2025-04-29 17:17:24,366 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
2025-04-29 17:17:25,013 - sentence_transformers.

In [8]:

from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
from llama_index.core.indices.property_graph import TextToCypherRetriever

graph_store = Neo4jPropertyGraphStore(
    username="neo4j",
    password="llamaindex",
    url="bolt://host.docker.internal:7687",
    database="neo4j"
)

graph_store.upsert_nodes(nodes)
graph_store.upsert_relations(relations)

kg_index = PropertyGraphIndex.from_existing(
    property_graph_store=graph_store
)


cypher_retriever = TextToCypherRetriever(
    kg_index.property_graph_store,
)

query_engine = kg_index.as_query_engine(
    similarity_top_k=10,
    embedding_mode="hybrid",
    response_mode="tree_summarize", #tree_summarize, no_text
    include_text=False,
    sub_retrievers=[cypher_retriever]
)

In [5]:
import nest_asyncio
nest_asyncio.apply()

query_engine.query("What was the Bitcoin Transaction Volume on 25th April 2025")

2025-04-29 16:16:26,026 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-04-29 16:16:26,950 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Response(response='There is no available data for the Bitcoin Transaction Volume on 25th April 2025.', source_nodes=[NodeWithScore(node=TextNode(id_='bfb5fb61-ce78-4664-8caf-95e3413aa955', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text="Generated Cypher query:\nMATCH (mv:MetricValue:Transactionvolumebtc {date: '2025-04-25'})\nRETURN mv.value AS bitcoin_transaction_volume\n\nCypher Response:\n[]", mimetype='text/plain', start_char_idx=None, end_char_idx=None, metadata_seperator='\n', text_template='{metadata_str}\n\n{content}'), score=1.0)], metadata={'bfb5fb61-ce78-4664-8caf-95e3413aa955': {}})

In [24]:
response = retriever.retrieve("What was the Bitcoin Transaction Volume on 25th April 2025")

2025-04-29 13:29:42,043 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/completions "HTTP/1.1 200 OK"


In [25]:
for node in response:
    print(node)

Node ID: 6154028b-f030-4c0b-acca-169737932c3d
Text: f2b560ecc28fe40ebbfe865359e6d06099aaa4dec2d7936daf1efa0bb925c0cf
-> SENDS_TO ->
bc1pq3qht4k45ccx89paqpd3axfwf4cmpyyxgwp79n6lxl8qv3hx8wuslyufll
Score:  0.000

Node ID: 1a42c6de-1f4e-4fc7-bc89-d0faf871ab0c
Text: transaction_volume_btc_2025-04-24 -> MEASURED_AT -> 2025-04-24
Score:  0.000

Node ID: ab947854-5322-47b4-844d-a404972734e4
Text: 69c7cfadeaaac47280c3f047bd31dc91c65b86431ac986f2f8de54b0ac3f70e9
-> CONTAINED_IN -> 894214
Score:  0.000

Node ID: 11848404-b9c0-4c56-801d-e13e0823b406
Text: 2025-03-30 -> HAS_METRIC -> active_addresses_2025-03-30
Score:  0.000

Node ID: cbf5ea26-5bbd-4f8d-bb30-3a388e16aba9
Text: dollar_index_2025-04-18 -> CORRELATES_WITH ->
difficulty_2025-04-18
Score:  0.000

Node ID: 02393098-d5e7-416d-9140-9b71af5f5142
Text: 894214 -> CONTAINS ->
c890ba545113a1d65b404f015adfc0756968afbed06a07d250384c930c4e895c
Score:  0.000

Node ID: 657e827b-c8c7-42c2-8f9f-3f9d2a91e864
Text: 96334648db516ba444180124aa7a4e314c1996